In [1]:
import json

with open("aiod_model.json", "r") as f:
    conceptual_model = json.load(f)

In [2]:
for i in range(len(conceptual_model["classes"])):
    if conceptual_model["classes"][i]["name"] == "Event":
        print(i)

54


In [3]:
conceptual_model["classes"][54]["inherited_properties"][0]
conceptual_model["classes"][54]["direct_properties"][0]

{'name': 'organiser',
 'domain': ['Event'],
 'range': ['Agent'],
 'annotations': {'comment': 'The entity organising the Event.',
  'label': 'organiser'},
 'equivalent_properties': []}

Generate schema

In [4]:
# Jupyter only looks in its default paths (sys.path does not automatically include sibling directories).
import sys
import os

# Define the absolute path to `aiod/src`
SRC_PATH = os.path.abspath("../src")

# Add it to Python's module search path if it's not already there
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

print(f"✅ Added {SRC_PATH} to sys.path")

✅ Added /home/taniya_das/Documents/AIOD-rest-api/src to sys.path


In [5]:
# Example schema generation
from database.model.ai_asset.ai_asset import AIAssetBase

json.loads(AIAssetBase.schema_json())
AIAssetBase.schema_json()

/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/taniya_das/Documents/AIOD-rest-api/venv/lib/python3.11/site-packages/sqlmodel/main.py:638: SAWarning: This declarative base already contains a class with the same class name and module name as database.model.helper_functions.LinkTable, and will be replaced in the string-lookup table.
  DeclarativeMeta.__init__(cls, classname, bases, dict_, **kw)


'{"title": "AIAssetBase", "type": "object", "properties": {"platform": {"title": "Platform", "description": "The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.", "maxLength": 64, "example": "example", "type": "string"}, "platform_resource_identifier": {"title": "Platform Resource Identifier", "description": "A unique identifier issued by the external platform that\'s specified in \'platform\'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.", "maxLength": 256, "example": "1", "type": "string"}, "name": {"title": "Name", "maxLength": 256, "example": "The name of this resource", "type": "string"}, "date_published": {"title": "Date Published", "description": "The datetime (utc) on which this resource was first published on an e

In [ ]:
"""
Step 1:
Extracts JSON schema from Pydantic/SQLModel classes and records parent classes.
- Handles errors per module.
"""

import os
import json
import importlib.util
from pathlib import Path
from pydantic import BaseModel
from sqlmodel import SQLModel

# base directory
BASE_DIR = "database/model"
OUTPUT_FILE = "schemas.json"

# dictionary to hold all schemas
all_schemas = {}
read_error = {}
class_parents = {}

for root, _, files in os.walk(os.path.join(SRC_PATH, BASE_DIR)):
    for file in files:
        if file.endswith(".py") and not file.startswith("__"):
            module_path = os.path.join(root, file)
            module_name = Path(module_path).with_suffix("").as_posix().split("/")[-1]

            try:
                spec = importlib.util.spec_from_file_location(module_name, module_path)
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)

                for attr_name in dir(module):
                    SQLModel.metadata.clear()

                    attr = getattr(module, attr_name)
                    # Only process Pydantic models (subclasses of BaseModel)
                    if (
                        isinstance(attr, type)
                        and issubclass(attr, BaseModel)
                        and attr is not BaseModel
                    ):
                        schema = attr.schema_json()
                        all_schemas[attr.__name__] = json.loads(schema)

                        # Get parent classes dynamically
                        parent_classes = [
                            parent.__name__
                            for parent in attr.__bases__
                            if isinstance(parent, type)
                            and issubclass(parent, BaseModel)
                            and parent is not BaseModel
                        ]

                        # Store parent mapping
                        class_parents[attr.__name__] = parent_classes

            except Exception as e:
                print(f"Error processing {module_name}: {e}")
                read_error.update({module_name: e})

Error processing location: issubclass() arg 1 must be a class


In [21]:
read_error

{'location': TypeError('issubclass() arg 1 must be a class')}

In [22]:
class_parents

{'NamedRelation': ['SQLModel'],
 'SQLModel': [],
 'AIoDConceptBase': ['SQLModel'],
 'Distribution': ['DistributionBase'],
 'DistributionBase': ['AIoDConceptBase'],
 'AIAssetTable': ['SQLModel'],
 'License': ['NamedRelation'],
 'AIAsset': ['AIAssetBase', 'AbstractAIResource'],
 'AIAssetBase': ['AIResourceBase'],
 'AIResourceBase': ['AIoDConceptBase'],
 'AbstractAIResource': ['AIResourceBase', 'AIoDConcept'],
 'Address': ['AddressBase'],
 'AddressBase': ['SQLModel'],
 'AddressORM': ['AddressBase'],
 'Geo': ['GeoBase'],
 'GeoBase': ['SQLModel'],
 'GeoORM': ['GeoBase'],
 'Language': ['NamedRelation'],
 'AgentTable': ['SQLModel'],
 'OrganisationType': ['NamedRelation'],
 'Agent': ['AgentBase', 'AbstractAIResource'],
 'AgentBase': ['AIResourceBase'],
 'Email': ['NamedRelation'],
 'Contact': ['ContactBase', 'AIoDConcept'],
 'Organisation': ['OrganisationBase', 'Agent'],
 'OrganisationBase': ['AgentBase'],
 'AIoDEntryORM': ['AIoDEntryBase'],
 'Expertise': ['NamedRelation'],
 'Person': ['Person

In [23]:
all_schemas

{'NamedRelation': {'title': 'NamedRelation',
  'description': 'An enumerable-type string (lowercase)',
  'type': 'object',
  'properties': {'identifier': {'title': 'Identifier', 'type': 'integer'},
   'name': {'title': 'Name',
    'description': 'The string value',
    'maxLength': 256,
    'type': 'string'}},
  'required': ['name']},
 'SQLModel': {'title': 'SQLModel', 'type': 'object', 'properties': {}},
 'AIoDConceptBase': {'title': 'AIoDConceptBase',
  'type': 'object',
  'properties': {'platform': {'title': 'Platform',
    'description': 'The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.',
    'maxLength': 64,
    'example': 'example',
    'type': 'string'},
   'platform_resource_identifier': {'title': 'Platform Resource Identifier',
    'description': "A unique identifier issued by the external platform that's specified in 'platform'. Leave empty

In [32]:
"""
Step 2:
Recursively converts all dictionary keys to lowercase.
Ensures "title" and "properties" fields are in lowercase.
"""


def convert_to_lowercase(data):
    """
    Convert dictionary keys and relevant values to lowercase without recursion.
    Works for:
    - class_parents (keys & values)
    - all_schemas (keys, titles, property names, and required fields)
    """
    if isinstance(data, dict):
        lowercased_data = {}

        for key, value in data.items():
            lower_key = key.lower()

            # Handle different structures (list for class_parents, dict for all_schemas)
            if isinstance(value, list):
                lowercased_data[lower_key] = [v.lower() for v in value]  # Convert list values
            elif isinstance(value, dict):
                lowercased_schema = {
                    "name": value.get("title", lower_key).lower(),  # Convert title
                    "description": value.get("description", ""),
                    "type": value.get("type", "").lower(),
                    "properties": {},
                    "required": [
                        field.lower() for field in value.get("required", [])
                    ],  # Lowercase required fields
                }

                # Convert properties inside schema
                if "properties" in value:
                    for prop_name, prop_details in value["properties"].items():
                        lower_prop_name = prop_name.lower()
                        lowercased_schema["properties"][lower_prop_name] = {
                            "name": prop_details.get(
                                "title", lower_prop_name
                            ).lower(),  # Convert property title
                            "title": prop_details.get(
                                "title", lower_prop_name
                            ).lower(),  # Ensure title is lowercase
                            **{
                                k: v for k, v in prop_details.items() if k != "title"
                            },  # Copy other attributes
                        }

                lowercased_data[lower_key] = lowercased_schema

        return lowercased_data

    return data  # Return as is if not a dict


# ✅ Apply conversion
class_parents = convert_to_lowercase(class_parents)  # Convert class parent mappings
all_schemas = convert_to_lowercase(all_schemas)  # Convert all schemas

In [ ]:
all_schemas["aiasset"]["properties"]

In [37]:
class_parents

{'namedrelation': ['sqlmodel'],
 'sqlmodel': [],
 'aiodconceptbase': ['sqlmodel'],
 'distribution': ['distributionbase'],
 'distributionbase': ['aiodconceptbase'],
 'aiassettable': ['sqlmodel'],
 'license': ['namedrelation'],
 'aiasset': ['aiassetbase', 'abstractairesource'],
 'aiassetbase': ['airesourcebase'],
 'airesourcebase': ['aiodconceptbase'],
 'abstractairesource': ['airesourcebase', 'aiodconcept'],
 'address': ['addressbase'],
 'addressbase': ['sqlmodel'],
 'addressorm': ['addressbase'],
 'geo': ['geobase'],
 'geobase': ['sqlmodel'],
 'geoorm': ['geobase'],
 'language': ['namedrelation'],
 'agenttable': ['sqlmodel'],
 'organisationtype': ['namedrelation'],
 'agent': ['agentbase', 'abstractairesource'],
 'agentbase': ['airesourcebase'],
 'email': ['namedrelation'],
 'contact': ['contactbase', 'aiodconcept'],
 'organisation': ['organisationbase', 'agent'],
 'organisationbase': ['agentbase'],
 'aiodentryorm': ['aiodentrybase'],
 'expertise': ['namedrelation'],
 'person': ['person

So in step 1, we used built in function to get schema. We also have class parents which are inherited classes. We then converted everything to lowercase. 
In step 2: we use class_parents dict, and add fields of the inherited classes into "properties" field of all_schemas. 

below is the same code. 

In [ ]:
"""
Step 3:
Expands class_parents dictionary and accounts for all indirect parents. 
all_inheritance stores complete parent hierarchy.
"""


def get_complete_inheritance(class_parents):
    all_inheritance = {
        key: set(value) for key, value in class_parents.items()
    }  # sets to avoid duplicates

    # Keep iterating until all inherited classes are fully expanded
    for key in class_parents:
        queue = list(class_parents[key])  # Start with direct parents

        while queue:
            parent = queue.pop(0)
            if parent in class_parents:
                new_parents = class_parents[parent]  # Get parent's parents
                queue.extend(new_parents)  # Add them to queue for further processing
                all_inheritance[key].update(new_parents)  # Add to final set

    # Convert sets back to lists for JSON compatibility
    all_inheritance = {k: list(v) for k, v in all_inheritance.items()}

    return all_inheritance


all_inheritance = get_complete_inheritance(class_parents)
all_inheritance

In [ ]:
"""
Step 4:
Add inherited classes field to `properties` of child class. 
Ensures no duplicates
"""


def add_inherited_properties(generated_schema, all_inheritance):
    """
    Adds inherited properties directly into 'properties' for each entity.
    """
    updated_schema = {}

    for entity_name, schema in generated_schema.items():
        # Copy the existing properties
        merged_properties = schema.get("properties")

        # If the entity has parent classes, inherit their properties
        for parent_class in all_inheritance.get(entity_name, []):
            # print(entity_name, parent_class)
            if parent_class in generated_schema:
                parent_properties = generated_schema[parent_class]["properties"]

                # Add missing properties from the parent class
                for prop_name, prop_details in parent_properties.items():
                    if prop_name not in merged_properties:  # Avoid duplicates
                        merged_properties[prop_name] = prop_details

        # Store the updated entity schema
        updated_schema[entity_name] = schema
        updated_schema[entity_name]["properties"] = (
            merged_properties  # Update properties with inherited ones
        )

    return updated_schema


schema_with_inherited = add_inherited_properties(all_schemas, all_inheritance)
schema_with_inherited

In [117]:
"""
Step 5:
Merges Base, Table, ORM, Type, Link classes into their main class.
Returns a new schema with merged properties.
"""


import re


def merge_base_classes(schema_with_inherited):
    merged_schema = {}
    class_mappings = {}

    # Identify base-like classes and map them to their main class
    for entity_name in schema_with_inherited:
        match = re.match(
            r"^(Abstract)?(.+?)(Base|Table|ORM|Type|Link)?$", entity_name, re.IGNORECASE
        )
        if match:
            core_name = match.group(2)  # The actual entity name
            main_class = core_name if core_name else entity_name
            class_mappings[entity_name] = main_class

    # Merge properties into main classes
    for entity_name, schema in schema_with_inherited.items():
        # normalized_name = entity_name
        main_entity_name = class_mappings.get(entity_name, entity_name)  # Resolve to main class
        # print(main_entity_name)

        if main_entity_name not in merged_schema:
            merged_schema[main_entity_name] = {
                "name": main_entity_name,
                "equivalent_classes": [],
                "properties": {},  # Will be renamed from "properties"
                "required": schema.get("required", []),
            }

        # Copy properties into direct_properties
        for prop_name, prop_details in schema.get("properties", {}).items():
            processed_property = prop_details
            processed_property["name"] = prop_name  # Rename "title" to "name"
            prop_name_cleaned = processed_property["name"]
            if "title" in processed_property:  # Remove "title" field if it exists
                del processed_property["title"]

            if prop_name_cleaned not in merged_schema[main_entity_name]["properties"]:
                merged_schema[main_entity_name]["properties"][prop_name_cleaned] = (
                    processed_property
                )

        # Merge required fields
        merged_schema[main_entity_name]["required"] = list(
            set(merged_schema[main_entity_name]["required"]) | set(schema.get("required", []))
        )

    return merged_schema


merged_schema = merge_base_classes(schema_with_inherited)
merged_schema["aiasset"]["properties"]

{'identifier': {'name': 'identifier', 'type': 'integer'},
 'type': {'name': 'type',
  'description': "The name of the table of the asset. E.g. 'organisation' or 'member'",
  'type': 'string'},
 'platform': {'name': 'platform',
  'description': 'The external platform from which this resource originates. Leave empty if this item originates from AIoD. If platform is not None, the platform_resource_identifier should be set as well.',
  'maxlength': 64,
  'example': 'example',
  'type': 'string'},
 'platform_resource_identifier': {'name': 'platform_resource_identifier',
  'description': "A unique identifier issued by the external platform that's specified in 'platform'. Leave empty if this item is not part of an external platform. For example, for HuggingFace, this should be the <namespace>/<dataset_name>, and for Openml, the OpenML identifier.",
  'maxlength': 256,
  'example': '1',
  'type': 'string'},
 'date_deleted': {'name': 'date_deleted',
  'type': 'string',
  'format': 'date-time'},

In [110]:
"""
Step 6:

"""


def add_inherited_properties(merged_schema, all_inheritance):
    """
    Step 3: Adds inherited properties after direct properties are fully merged.
    Uses `all_inheritance` to populate the `inherited_properties` field.
    """
    for entity_name, schema in merged_schema.items():
        inherited_properties = {}

        if entity_name in all_inheritance:
            for parent_class in all_inheritance[entity_name]:
                if parent_class in merged_schema:  # Ensure parent exists
                    parent_schema = merged_schema[parent_class]

                    for prop_name, prop_details in parent_schema["properties"].items():
                        prop_name_cleaned = prop_details["name"]

                        # Only add if it doesn't exist in direct_properties
                        if prop_name_cleaned not in schema["properties"]:
                            inherited_properties[prop_name_cleaned] = prop_details.copy()

        merged_schema[entity_name]["inherited_properties"] = (
            inherited_properties  # Add inherited properties
        )

    return merged_schema


# # Step 1: Merge base classes
# merged_schema = merge_base_classes(merged_schema)

# Step 3: Add inherited properties
preprocessed_generated_schema = add_inherited_properties(merged_schema, all_inheritance)

In [116]:
preprocessed_generated_schema["event"]["inherited_properties"].keys()

dict_keys([])

In [261]:
def add_inherited_properties(merged_schema, all_inheritance):
    """
    Step 3: Adds inherited properties after direct properties are fully merged.
    Uses `all_inheritance` to populate the `inherited_properties` field.
    """
    for entity_name, schema in merged_schema.items():
        inherited_properties = {}

        if entity_name in all_inheritance:
            for parent_class in all_inheritance[entity_name]:
                if parent_class in merged_schema:  # Ensure parent exists
                    parent_schema = merged_schema[parent_class]

                    for prop_name, prop_details in parent_schema["direct_properties"].items():
                        prop_name_cleaned = prop_details["name"]

                        # Only add if it doesn't exist in direct_properties
                        if prop_name_cleaned not in schema["direct_properties"]:
                            inherited_properties[prop_name_cleaned] = prop_details.copy()

        merged_schema[entity_name]["inherited_properties"] = (
            inherited_properties  # Add inherited properties
        )

    return merged_schema


# Step 1: Merge base classes
merged_schema = merge_base_classes(generated_schema)

# Step 3: Add inherited properties
preprocessed_generated_schema = add_inherited_properties(merged_schema, all_inheritance)

In [262]:
merged_schema.keys()

dict_keys(['NamedRelation', 'SQLModel', 'AIoDConcept', 'Distribution', 'AIAsset', 'License', 'AIResource', 'Address', 'Geo', 'Location', 'Language', 'Agent', 'Organisation', 'Email', 'Contact', 'AIoDEntry', 'Expertise', 'Person', 'Telephone', 'Team', 'ComputationalAsset', 'Publication', 'KnowledgeAsset', 'Text', 'AIoDEntryCreate', 'AIoDEntryRead', 'Body', 'AlternateName', 'ResearchArea', 'Keyword', 'IndustrialSector', 'AIResourcePart', 'AIResourceRelevant', 'ApplicationArea', 'Note', 'Relevant', 'ScientificDomain', 'News', 'NewsCategory', 'EventStatus', 'EventMode', 'Event', 'DatasetSize', 'Dataset', 'Badge', 'RunnableDistribution', 'Experiment', 'MLModel', 'Prerequisite', 'Pace', 'AccessMode', 'TargetAudience', 'EducationalLevel', 'EducationalResource', 'Platf', 'Platform', 'Service', 'Project', 'CaseStudy'])

In [140]:
import re

generated_schema = all_schemas.copy()


def preprocess_generated_schema(generated_schema, all_inheritance):
    """
    Preprocess the generated schema to:
    - Merge base classes (ending in Base, Table, ORM) into main classes
    - Keep the original class name (e.g., AIAsset stays AIAsset)
    - Copy properties from base classes to main classes without duplication
    - Rename "title" to "name"
    - Add "inherited_properties" using all_inheritance
    - Convert "properties" into "direct_properties" dictionary format
    - Ensure entity names are case-consistent
    """
    preprocessed_schema = {}
    merged_classes = {}  # Stores final merged entities
    original_case_map = {}  # To track original casing of entity names

    # Step 1: Identify mappings of base-like classes to main classes
    class_mappings = {}  # { "aiassetbase": "aiasset", "aiassettable": "aiasset", "aiassetorm": "aiasset" }

    for entity_name in generated_schema:
        normalized_name = entity_name.lower()
        original_case_map[normalized_name] = entity_name  # Store original case

        match = re.match(r"(.+)(Base|Table|ORM|Type|Link)$", entity_name, re.IGNORECASE)
        if match:
            main_class_name = match.group(1)  # Extract base name without suffix
            class_mappings[normalized_name] = main_class_name.lower()  # Store mapping

    # Step 2: Merge properties into the main class
    for entity_name, schema in generated_schema.items():
        normalized_name = entity_name.lower()

        # Determine the final entity name (handling merged classes)
        main_entity_name = class_mappings.get(normalized_name, normalized_name)

        # Use the original case for entity names
        final_entity_name = original_case_map.get(main_entity_name, main_entity_name)

        # Initialize main entity if not already added
        if final_entity_name not in merged_classes:
            merged_classes[final_entity_name] = {
                "name": final_entity_name,  # Keep original case
                "equivalent_classes": [],
                "direct_properties": {},  # Renamed from "properties"
                "inherited_properties": {},  # New field for inherited fields
                "required": schema.get("required", []),
            }

        # Copy direct properties, avoiding duplicates
        for prop_name, prop_details in schema.get("properties", {}).items():
            processed_property = prop_details.copy()
            processed_property["name"] = processed_property.pop(
                "title", prop_name
            )  # Rename "title" to "name"

            prop_name_cleaned = processed_property["name"].lower()

            # Add property to direct_properties if it doesn't exist
            if prop_name_cleaned not in merged_classes[final_entity_name]["direct_properties"]:
                merged_classes[final_entity_name]["direct_properties"][prop_name_cleaned] = (
                    processed_property
                )

        # Step 3: Add inherited properties
        inherited_properties = {}

        # Ensure the class exists in all_inheritance
        if entity_name in all_inheritance:
            # Iterate over all parent classes for the given entity
            for parent_class in all_inheritance[entity_name]:
                if parent_class in generated_schema:  # Check if parent class exists in the schema
                    parent_schema = generated_schema[parent_class]

                    # Ensure the parent class has properties
                    if "properties" in parent_schema:
                        for prop_name, prop_details in parent_schema["properties"].items():
                            # Ensure the property has a title
                            prop_name_cleaned = prop_details.get("title", prop_name).lower()

                            # Add property only if it does not exist in direct_properties
                            if (
                                prop_name_cleaned
                                not in merged_classes[final_entity_name]["direct_properties"]
                            ):
                                inherited_properties[prop_name_cleaned] = prop_details.copy()
                                inherited_properties[prop_name_cleaned]["name"] = prop_details.get(
                                    "title", prop_name
                                )

        merged_classes[final_entity_name]["inherited_properties"] = (
            inherited_properties  # Add inherited properties
        )

    return merged_classes  # Return as dictionary


# Apply preprocessing with inheritance
preprocessed_generated_schema = preprocess_generated_schema(generated_schema, all_inheritance)

In [182]:
preprocessed_generated_schema["KnowledgeAsset"]["inherited_properties"]

{}

In [129]:
def compare_schemas(conceptual_model, generated_schema):
    """
    Compare conceptual and generated schemas:
    - Identify missing and extra entities
    - Identify missing and extra properties for matching entities
    """
    # Convert both to case-insensitive dictionaries for easy lookup
    conceptual_dict = {entity["name"].lower(): entity for entity in conceptual_model}
    # generated_dict = {entity["name"].lower(): entity for entity in generated_schema}
    generated_dict = {key.lower(): value for key, value in generated_schema.items()}

    # Entities present in both schemas
    matching_entities = set(conceptual_dict.keys()) & set(generated_dict.keys())

    # Entities missing in generated schema
    missing_entities = set(conceptual_dict.keys()) - set(generated_dict.keys())

    # Entities extra in generated schema (not in conceptual model)
    extra_entities = set(generated_dict.keys()) - set(conceptual_dict.keys())

    # Compare properties for matching entities
    missing_properties = {}
    extra_properties = {}

    for entity_name in matching_entities:
        # Get conceptual model properties (merge direct & inherited properties)
        conceptual_properties = set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["direct_properties"]
        ) | set(
            prop["name"].lower() for prop in conceptual_dict[entity_name]["inherited_properties"]
        )

        # # Get generated schema properties
        # generated_properties = set(
        #     prop["name"].lower() for prop in generated_dict[entity_name]["direct_properties"]
        # )

        # Get generated schema properties
        generated_properties = set()

        if entity_name in generated_dict and "direct_properties" in generated_dict[entity_name]:
            generated_properties = {
                generated_dict[entity_name]["direct_properties"][prop]["name"].lower()
                for prop in generated_dict[entity_name]["direct_properties"]
            }

        # Identify missing and extra properties
        missing_props = conceptual_properties - generated_properties
        extra_props = generated_properties - conceptual_properties

        if missing_props:
            missing_properties[entity_name] = list(missing_props)

        if extra_props:
            extra_properties[entity_name] = list(extra_props)

    # Store results
    comparison_results = {
        "matching_entities": list(matching_entities),
        "missing_entities": list(missing_entities),
        "extra_entities": list(extra_entities),
        "missing_properties": missing_properties,
        "extra_properties": extra_properties,
    }

    return comparison_results


# Run comparison
comparison_results = compare_schemas(conceptual_model["classes"], preprocessed_generated_schema)

# # Save results to JSON
# with open("schema_comparison.json", "w") as f:
#     json.dump(comparison_results, f, indent=2)

# print("\n✅ Schema comparison completed! Results saved to 'schema_comparison.json'")

In [ ]:
comparison_results["missing_entities"]

In [151]:
def restructure_properties_single_row(properties_dict):
    """
    Restructures the properties dictionary into a DataFrame where:
    - Each entity is a row.
    - All its missing/extra properties are stored in the same row as separate columns.

    Parameters:
    - properties_dict: Dictionary with entities as keys and list of properties as values.

    Returns:
    - Pandas DataFrame
    """
    reformatted_data = [
        {"Entity": entity, **{f"Property {i + 1}": prop for i, prop in enumerate(props)}}
        for entity, props in properties_dict.items()
    ]

    return pd.DataFrame(reformatted_data)

In [166]:
import pandas as pd


def comparison_results_to_dataframe(comparison_results):
    """
    Convert comparison results dictionary into multiple Pandas DataFrames.

    Returns:
    - missing_entities_df: DataFrame listing missing entities.
    - extra_entities_df: DataFrame listing extra entities.
    - missing_properties_df: DataFrame listing missing properties per entity.
    - extra_properties_df: DataFrame listing extra properties per entity.
    """
    # Convert missing and extra entities to DataFrames
    missing_entities_df = pd.DataFrame(
        comparison_results.get("missing_entities", []), columns=["Missing Entities"]
    )
    extra_entities_df = pd.DataFrame(
        comparison_results.get("extra_entities", []), columns=["Extra Entities"]
    )

    # # Convert missing properties to DataFrame
    missing_properties_df = restructure_properties_single_row(
        comparison_results.get("missing_properties", {})
    )

    # Convert extra properties to DataFrame
    extra_properties_df = restructure_properties_single_row(
        comparison_results.get("extra_properties", {})
    )

    return missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df


missing_entities_df, extra_entities_df, missing_properties_df, extra_properties_df = (
    comparison_results_to_dataframe(comparison_results)
)

In [181]:
missing_properties_df[missing_properties_df["Entity"] == "knowledgeasset"].dropna(axis=1)

,Entity,Property 1,Property 2,Property 3,Property 4,Property 5,Property 6,Property 7,Property 8,Property 9,...,Property 13,Property 14,Property 15,Property 16,Property 17,Property 18,Property 19,Property 20,Property 21,Property 22
0,knowledgeasset,creator,part_of,has_part,media,alternate_name,distribution,applies,description,licence,...,documents,keyword,sameas,date_published,contact,aiod_entry,pricing,business_sector,location,date_created


In [175]:
for i in range(len(conceptual_model["classes"])):
    if conceptual_model["classes"][i]["name"] == "KnowledgeAsset":
        print(i)

65


In [172]:
# missing_entities_df
# extra_entities_df
missing_properties_df
# # extra_properties_df

,Entity,Property 1,Property 2,Property 3,Property 4,Property 5,Property 6,Property 7,Property 8,Property 9,...,Property 21,Property 22,Property 23,Property 24,Property 25,Property 26,Property 27,Property 28,Property 29,Property 30
0,knowledgeasset,creator,part_of,has_part,media,alternate_name,distribution,applies,description,licence,...,location,date_created,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,airesource,media,alternate_name,keyword,contact,aiod_entry,sameas,business_sector,description,part_of,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,researcharea,relevant_link,sameas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,agent,media,alternate_name,keyword,sameas,contact,aiod_entry,business_sector,description,part_of,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,distribution,sameas,checksum_algorithm,content_url,date_modified,identifier,relevant_link,has_computational_requirement,technology_readiness_level,encoding_format,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,experiment,experimental_workflow,uses_model,creator,part_of,uses_dataset,has_part,media,alternate_name,reproducibility_explanation,...,date_published,contact,aiod_entry,pricing,business_sector,location,date_created,NaN,NaN,NaN
6,aiodentry,editor,sameas,modified,relevant_link,entry_created,content,entry_published,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,dataset,creator,part_of,has_part,pid,variable_measured,media,alternate_name,distribution,applies,...,contact,aiod_entry,pricing,business_sector,location,funder,measurement_technique,date_created,NaN,NaN
8,newscategory,relevant_link,sameas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,event,part_of,has_part,event_mode,media,alternate_name,description,relevant_link,research_area,documented_in,...,location,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [171]:
# save the df in a csv

missing_properties_df.to_csv("missing_properties.csv", index=False)
missing_entities_df.to_csv("missing_entities.csv", index=False)
extra_entities_df.to_csv("extra_entities.csv", index=False)
extra_properties_df.to_csv("extra_properties.csv", index=False)